# Feature Engineering — Climate Risk & Health

Toutes les features créées sont justifiées par une hypothèse épidémiologique.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../data')
SEED = 42

train   = pd.read_csv(DATA / 'Train.csv', parse_dates=['deathdate'])
test    = pd.read_csv(DATA / 'Test.csv',  parse_dates=['deathdate'])
climate = pd.read_csv(DATA / 'climate_features.csv', parse_dates=['deathdate'])

print(f'Train: {train.shape}, Test: {test.shape}, Climate: {climate.shape}')

Train: (3146, 13), Test: (1030, 12), Climate: (4176, 18)


## 1. Merge avec les features climatiques

In [2]:
climate_feats = climate.drop(columns=['deathdate'])

train = train.merge(climate_feats, on='ID', how='left')
test  = test.merge(climate_feats,  on='ID', how='left')

print(f'Train merged: {train.shape}')
print(f'Test  merged: {test.shape}')

Train merged: (3146, 29)
Test  merged: (1030, 28)


## 2. Features temporelles

Hypothèse : la saison modère le risque — deux saisons des pluies en Ouganda (mars-mai et oct-nov) → pics de paludisme.

In [3]:
def add_temporal_features(df):
    df = df.copy()
    df['month']       = df['deathdate'].dt.month
    df['day_of_year'] = df['deathdate'].dt.dayofyear
    df['year']        = df['deathdate'].dt.year

    # Saisons Ouganda : Long Rains (mar-mai=1), Long Dry (jun-jul=2),
    #                   Short Rains (aug-nov=3), Short Dry (dec-fev=4)
    season_map = {
        1: 4, 2: 4,
        3: 1, 4: 1, 5: 1,
        6: 2, 7: 2,
        8: 3, 9: 3, 10: 3, 11: 3,
        12: 4
    }
    df['season_uganda'] = df['month'].map(season_map)

    # Cyclicité du mois (sin/cos pour éviter la discontinuité jan-déc)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Cyclicité du jour de l'année
    df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    return df

train = add_temporal_features(train)
test  = add_temporal_features(test)
print('Features temporelles ajoutées :', ['month','day_of_year','year','season_uganda','month_sin','month_cos','doy_sin','doy_cos'])

Features temporelles ajoutées : ['month', 'day_of_year', 'year', 'season_uganda', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']


## 3. Features d'âge

Hypothèse : l'âge est non-linéaire et bimodal. Les enfants < 5 ans meurent surtout de malaria/diarrhée (climate-sensitive). Les adultes > 60 ans meurent surtout de maladies chroniques (non climate-sensitive).

In [4]:
def add_age_features(df):
    df = df.copy()

    # Flags binaires (hypothèses épidémiologiques)
    df['is_neonate']    = (df['age'] < 1).astype(int)   # < 1 an : infections périnatales
    df['is_infant']     = (df['age'] < 2).astype(int)   # < 2 ans
    df['is_child_u5']   = (df['age'] < 5).astype(int)   # < 5 ans : OMS seuil mortalité infanto-juvénile
    df['is_child']      = ((df['age'] >= 1) & (df['age'] < 15)).astype(int)
    df['is_adult']      = ((df['age'] >= 15) & (df['age'] < 60)).astype(int)
    df['is_elderly']    = (df['age'] >= 60).astype(int)

    # Transformation log (compresse la longue queue des adultes/âgés)
    df['age_log'] = np.log1p(df['age'])

    # Buckets numériques (pour les modèles linéaires)
    df['age_bucket'] = pd.cut(
        df['age'],
        bins=[0, 1, 5, 15, 60, 120],
        labels=[0, 1, 2, 3, 4],
        right=False
    ).astype(int)

    return df

train = add_age_features(train)
test  = add_age_features(test)
print('Features âge ajoutées.')

Features âge ajoutées.


## 4. Features climatiques dérivées

Hypothèse : les anomalies et intensités ont plus de sens que les valeurs absolues.

In [5]:
def add_climate_features(df):
    df = df.copy()

    # Amplitude thermique journalière (stress thermique)
    df['temp_range_daily'] = df['max_temperature'] - df['min_temperature']

    # Anomalie pluie court terme vs long terme (accélération des pluies)
    df['rain_anomaly_7v30']  = df['rain_sum_7d']  - df['rain_sum_30d']  / (30/7)
    df['rain_anomaly_30v90'] = df['rain_sum_30d'] - df['rain_sum_90d']  / 3

    # Anomalie température court terme vs long terme (rupture climatique)
    df['temp_anomaly_7v90']  = df['tavg_7d']  - df['tavg_90d']
    df['temp_anomaly_30v90'] = df['tavg_30d'] - df['tavg_90d']

    # Intensité des pluies (pluies intenses = contamination eau, vecteurs)
    df['rain_intensity_30d'] = np.where(
        df['rain_days_30d'] > 0,
        df['rain_sum_30d'] / df['rain_days_30d'],
        0
    )

    # Flag période humide (>5mm/jour en moyenne sur 7j)
    df['is_rainy_week'] = (df['rain_sum_7d'] > 35).astype(int)

    # Flag pluie le jour du décès (données brutes)
    df['is_rainy_day'] = (df['precipitation'] > 0).astype(int)

    # NDVI : végétation luxuriante = habitat moustique
    df['ndvi_delta'] = df['ndvi_30d'] - df['ndvi_90d']

    # Interaction NDVI × pluie (conditions favorables à Anopheles)
    df['ndvi_x_rain30'] = df['ndvi_30d'] * df['rain_sum_30d']

    # ── Features basées sur la littérature (Iganga-Mayuge HDSS) ──────────────
    # Seuil critique pluie : >200mm/semaine = risque malaria (PMC12676583)
    df['is_heavy_rain_week'] = (df['rain_sum_7d'] > 200).astype(int)

    # Saison pic paludisme Ouganda (lag ~4-6 sem après saisons des pluies)
    # Pics : mai-juillet (post Long Rains) et novembre-janvier (post Short Rains)
    df['is_malaria_peak_season'] = df['month'].isin([5, 6, 7, 11, 12, 1]).astype(int)

    # Température dans la plage optimale transmission Plasmodium (20-32°C)
    df['temp_in_malaria_range'] = (
        (df['tavg_30d'] >= 20) & (df['tavg_30d'] <= 32)
    ).astype(int)
    # ─────────────────────────────────────────────────────────────────────────

    return df

train = add_climate_features(train)
test  = add_climate_features(test)
print('Features climatiques dérivées ajoutées.')

Features climatiques dérivées ajoutées.


## 5. Interactions âge × climat

Hypothèse : un nourrisson sous forte pluie = risque extrême de malaria. Un adulte sous les mêmes conditions = risque modéré.

In [6]:
def add_interactions(df):
    df = df.copy()

    # Nourrisson × conditions propices au paludisme
    df['u5_x_rain30']    = df['is_child_u5'] * df['rain_sum_30d']
    df['u5_x_ndvi']      = df['is_child_u5'] * df['ndvi_30d']
    df['u5_x_rainy_wk']  = df['is_child_u5'] * df['is_rainy_week']

    # Âge log × pluie (effet continu)
    df['age_log_x_rain30'] = df['age_log'] * df['rain_sum_30d']
    df['age_log_x_tavg30'] = df['age_log'] * df['tavg_30d']

    # Âgé × anomalie de chaleur (vague de chaleur → mortalité cardiovasculaire)
    df['elderly_x_temp_anom'] = df['is_elderly'] * df['temp_anomaly_7v90']

    # Saison des pluies × enfant
    df['rain_season_x_u5'] = df['is_child_u5'] * (df['season_uganda'].isin([1, 3])).astype(int)

    return df

train = add_interactions(train)
test  = add_interactions(test)
print('Interactions ajoutées.')

Interactions ajoutées.


## 6. Nettoyage final — features à exclure

In [7]:
DROP_COLS = [
    'location',      # quasi-aucun overlap train/test (fuite si encodé)
    'hot_days_30d',  # variance nulle (= 0 partout)
    'slope',         # corrélé à elevation à 0.993 — redondant
    'deathdate',     # remplacée par month, day_of_year, year
    'ID',            # identifiant
]

TARGET = 'is_climate_sensitive'

feature_cols = [c for c in train.columns if c not in DROP_COLS + [TARGET]]

print(f'Nb features finales : {len(feature_cols)}')
print()
print('Liste des features :')
for c in feature_cols:
    print(f'  {c}')

Nb features finales : 59

Liste des features :
  zone
  gender
  age
  avg_temperature
  max_temperature
  min_temperature
  precipitation
  latitude
  longitude
  elevation
  max_daily_rain_30d
  ndvi_30d
  ndvi_90d
  rain_days_30d
  rain_sum_30d
  rain_sum_7d
  rain_sum_90d
  tavg_30d
  tavg_7d
  tavg_90d
  temp_range_mean_30d
  tmax_30d
  tmin_30d
  month
  day_of_year
  year
  season_uganda
  month_sin
  month_cos
  doy_sin
  doy_cos
  is_neonate
  is_infant
  is_child_u5
  is_child
  is_adult
  is_elderly
  age_log
  age_bucket
  temp_range_daily
  rain_anomaly_7v30
  rain_anomaly_30v90
  temp_anomaly_7v90
  temp_anomaly_30v90
  rain_intensity_30d
  is_rainy_week
  is_rainy_day
  ndvi_delta
  ndvi_x_rain30
  is_heavy_rain_week
  is_malaria_peak_season
  temp_in_malaria_range
  u5_x_rain30
  u5_x_ndvi
  u5_x_rainy_wk
  age_log_x_rain30
  age_log_x_tavg30
  elderly_x_temp_anom
  rain_season_x_u5


## 7. Sauvegarde des datasets traités

In [ ]:
# Sauvegarde des IDs pour la soumission
test_ids = test['ID'].copy()

X_train = train[feature_cols].copy()
y_train = train[TARGET]
X_test  = test[feature_cols].copy()

# Encoder les catégorielles (zone, gender)
cat_cols = ['zone', 'gender']
print('Colonnes catégorielles :', cat_cols)

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')

print(f'\nX_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}, équilibre = {y_train.mean():.3f}')

# Sauvegarde
X_train.to_parquet(DATA / 'X_train.parquet')
y_train.to_frame().to_parquet(DATA / 'y_train.parquet')
X_test.to_parquet(DATA / 'X_test.parquet')
test_ids.to_frame().to_parquet(DATA / 'test_ids.parquet')

print('\nDatasets sauvegardés dans data/')

In [9]:
# Vérification : pas de NaN dans les features finales
nan_train = X_train.isnull().sum().sum()
nan_test  = X_test.isnull().sum().sum()
print(f'NaN X_train : {nan_train}')
print(f'NaN X_test  : {nan_test}')

# Vérification overlap ID train/test
assert len(set(train['ID']) & set(test['ID'])) == 0, 'FUITE : IDs communs train/test !'
print('✓ Aucun ID commun train/test')
print('✓ Feature engineering complet')

NaN X_train : 0
NaN X_test  : 0
✓ Aucun ID commun train/test
✓ Feature engineering complet
